## Continued Pretraining Phase

In [ ]:
from src.core.settings import settings

CPT_DATASET_SEED = 42
INS_FT_DATASET_SEED = 123

In [ ]:
from src.get_training_data.pretraining_data import download_multi_datasets_async, split_and_save_cpt_dataset

In [ ]:
cpt_dataset = await download_multi_datasets_async(seed = CPT_DATASET_SEED)

print(cpt_dataset)

cpt_dataset = await split_and_save_cpt_dataset(cpt_dataset)

print(cpt_dataset)

In [ ]:
from src.core.training.cpt import get_model_and_tokenizer_async, tokenize_dataset

In [ ]:
model, tokenizer = await get_model_and_tokenizer_async()

In [ ]:
cpt_train_data = cpt_dataset['train']
cpt_val_data = cpt_dataset['legal_val']

In [ ]:
cpt_train_data_tokenized = tokenize_dataset(cpt_train_data)
cpt_val_data_tokenized = tokenize_dataset(cpt_val_data)

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArguments

MAX_SEQ_LEN_CPT = settings.MAX_SEQ_LEN_CPT

trainer = UnslothTrainer(
    model = model,
    train_dataset = cpt_train_data_tokenized,
    eval_dataset=cpt_val_data_tokenized,
    tokenizer = tokenizer,
    args = UnslothTrainingArguments(
        max_seq_length = MAX_SEQ_LEN_CPT,
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 4,
        learning_rate = 5e-5,
        embedding_learning_rate = 5e-6,
        warmup_steps = 10,
        dataset_num_proc = 8,
        max_steps = 100,
        # save_steps = 25,
        logging_steps = 10,
        output_dir = "outputs",
        optim = "adamw_8bit",
        seed = 3407,
        fp16 = True,
        bf16 = False
    ),
)

trainer.train()